# Data processing - shots

In [21]:
import pandas as pd

DATA_PATH = "../../data/"

In [29]:
def get_season(date): # TODO: using gameId instead?
    year = date.year
    if date.month >= 9:
        return f"{year}-{year+1}"
    else:
        return f"{year-1}-{year}"

## Load shot detail data

In [23]:
import glob

shot_files = sorted(glob.glob(DATA_PATH + "/nba_play_by_play_shot_data/shotdetail*.csv"))

len(shot_files)

58

## Process data

In [24]:
# GRID_TYPE,GAME_ID,GAME_EVENT_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_NAME,PERIOD,MINUTES_REMAINING,SECONDS_REMAINING,
# EVENT_TYPE,ACTION_TYPE,SHOT_TYPE,SHOT_ZONE_BASIC,SHOT_ZONE_AREA,SHOT_ZONE_RANGE,SHOT_DISTANCE,LOC_X,LOC_Y,SHOT_ATTEMPTED_FLAG,SHOT_MADE_FLAG,GAME_DATE,HTM,VTM

# hoop is at (LOC_X, LOC_Y) = (0, 0)

cols_to_keep = [
    "GAME_ID", "GAME_DATE", "PLAYER_ID", "PLAYER_NAME", "TEAM_ID", "TEAM_NAME",
    "PERIOD", "MINUTES_REMAINING", "SECONDS_REMAINING",
    "SHOT_TYPE", "SHOT_DISTANCE", "LOC_X", "LOC_Y",
    "SHOT_ATTEMPTED_FLAG", "SHOT_MADE_FLAG"
]

frames = []

for file_path in shot_files:
    df = pd.read_csv(file_path, usecols=lambda c: c in cols_to_keep)

    # 1 = actual shot attempt
    if "SHOT_ATTEMPTED_FLAG" in df.columns:
        df = df[df["SHOT_ATTEMPTED_FLAG"] == 1]

    df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"], format="%Y%m%d", errors="coerce")
    df = df[df["GAME_DATE"].notna()]
    df["season"] = df["GAME_DATE"].apply(get_season)

    # "_po_" = Playoffs
    df["gameType"] = "Playoffs" if "_po_" in file_path.lower() else "Regular Season"

    # cleanup corrdinates
    df["LOC_X"] = pd.to_numeric(df["LOC_X"], errors="coerce")
    df["LOC_Y"] = pd.to_numeric(df["LOC_Y"], errors="coerce")
    df = df.dropna(subset=["LOC_X", "LOC_Y"])

    frames.append(df)

shot_events = pd.concat(frames, ignore_index=True)

shot_events = shot_events.rename(columns={
    "GAME_ID": "gameId",
    "GAME_DATE": "gameDate",
    "PLAYER_ID": "personId",
    "PLAYER_NAME": "playerName",
    "TEAM_ID": "teamId",
    "TEAM_NAME": "teamName",
    "PERIOD": "period",
    "MINUTES_REMAINING": "minutesRemaining",
    "SECONDS_REMAINING": "secondsRemaining",
    "SHOT_TYPE": "shotType",
    "SHOT_DISTANCE": "shotDistance",
    "LOC_X": "locX",
    "LOC_Y": "locY",
    "SHOT_MADE_FLAG": "shotMade",
})

# SHOT_TYPE: 3PT Field Goal or 2PT Field Goal
shot_events["shotValue"] = shot_events["shotType"].str.extract(r"(\d)PT")[0].astype("Int64")

shot_events = shot_events[[
    "season", "gameType", "gameId", "gameDate",
    "personId", "playerName", "teamId", "teamName",
    "period", "minutesRemaining", "secondsRemaining",
    "shotType", "shotValue", "shotDistance", "locX", "locY", "shotMade"
]]

shot_events.head()

,season,gameType,gameId,gameDate,personId,playerName,teamId,teamName,period,minutesRemaining,secondsRemaining,shotType,shotValue,shotDistance,locX,locY,shotMade
0,1996-1997,Regular Season,29600005,1996-11-01,120,Steven Smith,1610612737,Atlanta Hawks,1,11,8,2PT Field Goal,2,0,0,0,1
1,1996-1997,Regular Season,29600005,1996-11-01,120,Steven Smith,1610612737,Atlanta Hawks,1,10,32,2PT Field Goal,2,16,153,71,0
2,1996-1997,Regular Season,29600005,1996-11-01,363,Christian Laettner,1610612737,Atlanta Hawks,1,9,38,2PT Field Goal,2,10,-103,26,1
3,1996-1997,Regular Season,29600005,1996-11-01,120,Steven Smith,1610612737,Atlanta Hawks,1,8,57,2PT Field Goal,2,0,0,0,0
4,1996-1997,Regular Season,29600005,1996-11-01,87,Dikembe Mutombo,1610612737,Atlanta Hawks,1,8,56,2PT Field Goal,2,0,0,0,0


### Check range for locX, locY

In [27]:
print("locX:", (shot_events["locX"].min(), shot_events["locX"].max()))
print("locY:", (shot_events["locY"].min(), shot_events["locY"].max()))

locX: (np.int64(-250), np.int64(250))
locY: (np.int64(-52), np.int64(884))


## Save data

In [28]:
shot_events.to_csv(DATA_PATH + "/processed/shot_events.csv", index=False)